# Retry-safe Microsoft Fabric people-counting pipeline

This notebook processes exactly one video per Fabric Notebook activity. Frames stay sequential because tracker state depends on frame order; scale out by running separate notebook activities for separate videos.

Retries are safe when every retry receives the same `RUN_ID` and inputs:

- an input SHA-256 and canonical configuration hash prevent accidental reuse of a run ID;
- a matching `SUCCEEDED` run exits without repeating inference;
- successful rows are replaced by scoped Delta merges, including deletion of stale rows;
- failed attempts write only attempt-scoped diagnostic snapshots, never authoritative output;
- the run ledger is marked `SUCCEEDED` only after both authoritative tables converge.

A failure is always re-raised so the Fabric pipeline activity can apply its retry policy.

## 1. Deploy and configure Fabric

1. Build the matching bundle from the repository root with `uv run python scripts/build_sdk_bundle.py cpu` or `gpu`.
2. Upload every wheel from the bundle's `wheels/` directory to a Fabric Environment, publish it, and attach it to this notebook.
3. Attach a default Lakehouse. The identity running the notebook needs read access to the source and create/write access to the target Delta tables.
4. In the Fabric Data Pipeline Notebook activity, add base parameters whose names match the parameter cell below. Set `RUN_ID` to `@pipeline().RunId` and set `VIDEO_URI` to the activity's video path.
5. Configure activity retries as needed, but do not generate a new run ID inside a retry. To process many videos concurrently, fan out to one notebook activity per video with a distinct stable run ID.

Prefer `/lakehouse/default/Files/...` because OpenCV can read it directly. An `abfs://` or `abfss://` source is copied to attempt-unique notebook-local storage before inference.

In [ ]:
# Fabric Pipeline base parameters override these values by name.
RUN_ID = ""
VIDEO_URI = "/lakehouse/default/Files/incoming/entrance-camera.mp4"
PIPELINE = "rtdetr-osnet"  # rtdetr-osnet or rfdetr-botsort
DEVICE_VARIANT = "cpu"  # cpu or gpu; must match the attached bundle
DEVICE = "cpu"
BATCH_SIZE = 1
SAMPLE_FPS = 3.0  # Use None to process every source frame.
DETECTION_THRESHOLD = 0.6
USE_FP16 = False
LINE = []  # Empty, or [x1, y1, x2, y2] in source-video pixels.
DETECTOR_MODEL = "r18"  # RT-DETR/OSNet only: r18 or r50
CAMERA_MOTION_COMPENSATION = None  # RF-DETR/BoT-SORT only
DATABASE = ""  # Empty uses the attached Lakehouse's default database.
TABLE_PREFIX = "people_counter"

## 2. Validate parameters and define the execution contract

Fabric may deliver base parameters as strings. The parsers below accept the corresponding native values or strict string representations and reject ambiguous input. Table identifiers are restricted before they are interpolated into Spark SQL names.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import urlparse
import hashlib
import importlib
import json
import re
import uuid

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession, functions as F, types as T

from people_counter import (
    RFDetrBotsortConfig,
    RTDetrOsnetConfig,
    line_count_records,
    run,
    telemetry_records,
)


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
RUN_ID_MAX_LENGTH = 200


@dataclass(frozen=True)
class Settings:
    run_id: str
    video_uri: str
    pipeline: str
    device_variant: str
    device: str
    batch_size: int
    sample_fps: float | None
    detection_threshold: float
    use_fp16: bool
    line: tuple[int, int, int, int] | None
    detector_model: str
    camera_motion_compensation: bool | None
    database: str
    table_prefix: str

    def canonical_config(self) -> dict[str, Any]:
        return {
            "video_uri": self.video_uri,
            "pipeline": self.pipeline,
            "device_variant": self.device_variant,
            "device": self.device,
            "batch_size": self.batch_size,
            "sample_fps": self.sample_fps,
            "detection_threshold": self.detection_threshold,
            "use_fp16": self.use_fp16,
            "line": self.line,
            "detector_model": self.detector_model,
            "camera_motion_compensation": self.camera_motion_compensation,
        }


def require_text(value: object, name: str, *, max_length: int | None = None) -> str:
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{name} must be a non-empty string")
    parsed = value.strip()
    if max_length is not None and len(parsed) > max_length:
        raise ValueError(f"{name} must contain at most {max_length} characters")
    return parsed


def parse_bool(value: object, name: str) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"
    raise ValueError(f"{name} must be true or false")


def parse_optional_bool(value: object, name: str) -> bool | None:
    if value is None or (isinstance(value, str) and value.strip().lower() in {"", "none", "null"}):
        return None
    return parse_bool(value, name)


def parse_int(value: object, name: str) -> int:
    if isinstance(value, bool):
        raise ValueError(f"{name} must be an integer")
    try:
        parsed = int(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must be an integer") from error
    if isinstance(value, float) and not value.is_integer():
        raise ValueError(f"{name} must be an integer")
    return parsed


def parse_optional_float(value: object, name: str) -> float | None:
    if value is None or (isinstance(value, str) and value.strip().lower() in {"", "none", "null"}):
        return None
    if isinstance(value, bool):
        raise ValueError(f"{name} must be numeric or null")
    try:
        return float(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must be numeric or null") from error


def parse_line(value: object) -> tuple[int, int, int, int] | None:
    if value is None or value == [] or (isinstance(value, str) and value.strip().lower() in {"", "none", "null", "[]"}):
        return None
    parsed = json.loads(value) if isinstance(value, str) else value
    if not isinstance(parsed, (list, tuple)) or len(parsed) != 4:
        raise ValueError("LINE must be empty or contain exactly four integers")
    coordinates = tuple(parse_int(item, "LINE coordinate") for item in parsed)
    return coordinates[0], coordinates[1], coordinates[2], coordinates[3]


def validate_identifier(value: object, name: str, *, allow_empty: bool = False) -> str:
    if allow_empty and isinstance(value, str) and not value.strip():
        return ""
    parsed = require_text(value, name)
    if IDENTIFIER.fullmatch(parsed) is None:
        raise ValueError(f"{name} must contain only letters, digits, and underscores and cannot start with a digit")
    return parsed


def load_settings() -> Settings:
    pipeline = require_text(PIPELINE, "PIPELINE")
    if pipeline not in {"rtdetr-osnet", "rfdetr-botsort"}:
        raise ValueError("PIPELINE must be rtdetr-osnet or rfdetr-botsort")
    device_variant = require_text(DEVICE_VARIANT, "DEVICE_VARIANT")
    if device_variant not in {"cpu", "gpu"}:
        raise ValueError("DEVICE_VARIANT must be cpu or gpu")
    device = require_text(DEVICE, "DEVICE")
    use_fp16 = parse_bool(USE_FP16, "USE_FP16")
    if device_variant == "cpu" and device != "cpu":
        raise ValueError("The CPU bundle requires DEVICE=cpu")
    if device_variant == "cpu" and use_fp16:
        raise ValueError("USE_FP16 cannot be enabled for the CPU bundle")
    batch_size = parse_int(BATCH_SIZE, "BATCH_SIZE")
    if batch_size < 1:
        raise ValueError("BATCH_SIZE must be at least 1")
    sample_fps = parse_optional_float(SAMPLE_FPS, "SAMPLE_FPS")
    if sample_fps is not None and sample_fps <= 0:
        raise ValueError("SAMPLE_FPS must be greater than zero or null")
    detection_threshold = parse_optional_float(DETECTION_THRESHOLD, "DETECTION_THRESHOLD")
    if detection_threshold is None or not 0.0 <= detection_threshold <= 1.0:
        raise ValueError("DETECTION_THRESHOLD must be between 0 and 1")
    detector_model = require_text(DETECTOR_MODEL, "DETECTOR_MODEL")
    if detector_model not in {"r18", "r50"}:
        raise ValueError("DETECTOR_MODEL must be r18 or r50")
    return Settings(
        run_id=require_text(RUN_ID, "RUN_ID", max_length=RUN_ID_MAX_LENGTH),
        video_uri=require_text(VIDEO_URI, "VIDEO_URI"),
        pipeline=pipeline,
        device_variant=device_variant,
        device=device,
        batch_size=batch_size,
        sample_fps=sample_fps,
        detection_threshold=detection_threshold,
        use_fp16=use_fp16,
        line=parse_line(LINE),
        detector_model=detector_model,
        camera_motion_compensation=parse_optional_bool(
            CAMERA_MOTION_COMPENSATION,
            "CAMERA_MOTION_COMPENSATION",
        ),
        database=validate_identifier(DATABASE, "DATABASE", allow_empty=True),
        table_prefix=validate_identifier(TABLE_PREFIX, "TABLE_PREFIX"),
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        while chunk := source.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_json(value: dict[str, Any]) -> tuple[str, str]:
    canonical = json.dumps(value, sort_keys=True, separators=(",", ":"))
    return canonical, hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def table_name(settings: Settings, suffix: str) -> str:
    name = f"{settings.table_prefix}_{suffix}"
    return f"{settings.database}.{name}" if settings.database else name


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Microsoft Fabric Spark session is required")
spark_session = spark_candidate

## 3. Define Delta schemas and retry-safe persistence

The explicit schemas make empty result sets safe. `replace_scope_rows` merges the current attempt and removes stale rows from an earlier attempt within the same run or failed-attempt scope.

In [ ]:
from datetime import datetime, timezone


RUN_SCHEMA = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("attempt_id", T.StringType(), False),
    T.StructField("source_uri", T.StringType(), False),
    T.StructField("source_video", T.StringType(), False),
    T.StructField("input_sha256", T.StringType(), False),
    T.StructField("config_json", T.StringType(), False),
    T.StructField("config_sha256", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("started_at", T.TimestampType(), False),
    T.StructField("completed_at", T.TimestampType(), True),
    T.StructField("distinct_people", T.LongType(), True),
    T.StructField("line_in_count", T.LongType(), True),
    T.StructField("line_out_count", T.LongType(), True),
    T.StructField("processed_frames", T.LongType(), True),
    T.StructField("ended_early", T.BooleanType(), True),
    T.StructField("error_type", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
])

PROVENANCE_FIELDS = [
    T.StructField("run_id", T.StringType(), False),
    T.StructField("attempt_id", T.StringType(), False),
    T.StructField("source_uri", T.StringType(), False),
    T.StructField("source_video", T.StringType(), False),
    T.StructField("input_sha256", T.StringType(), False),
    T.StructField("config_sha256", T.StringType(), False),
    T.StructField("recorded_at", T.TimestampType(), False),
]

TELEMETRY_SCHEMA = T.StructType(PROVENANCE_FIELDS + [
    T.StructField("person_id", T.LongType(), False),
    T.StructField("entry_frame", T.LongType(), False),
    T.StructField("exit_frame", T.LongType(), False),
    T.StructField("entry_seconds", T.DoubleType(), False),
    T.StructField("exit_seconds", T.DoubleType(), False),
    T.StructField("entry_timestamp", T.StringType(), False),
    T.StructField("exit_timestamp", T.StringType(), False),
    T.StructField("duration_seconds", T.DoubleType(), False),
])

LINE_COUNT_SCHEMA = T.StructType(PROVENANCE_FIELDS + [
    T.StructField("frame", T.LongType(), False),
    T.StructField("video_seconds", T.StringType(), False),
    T.StructField("video_timestamp", T.StringType(), False),
    T.StructField("frame_in_count", T.LongType(), False),
    T.StructField("frame_out_count", T.LongType(), False),
    T.StructField("cumulative_in_count", T.LongType(), False),
    T.StructField("cumulative_out_count", T.LongType(), False),
    T.StructField("line_start_x", T.LongType(), False),
    T.StructField("line_start_y", T.LongType(), False),
    T.StructField("line_end_x", T.LongType(), False),
    T.StructField("line_end_y", T.LongType(), False),
])


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def ensure_tables(settings: Settings) -> dict[str, str]:
    if settings.database:
        spark_session.sql(f"CREATE DATABASE IF NOT EXISTS `{settings.database}`")
    tables = {
        "runs": table_name(settings, "runs"),
        "telemetry": table_name(settings, "telemetry"),
        "line_counts": table_name(settings, "line_counts"),
        "failed_telemetry": table_name(settings, "failed_attempt_telemetry"),
        "failed_line_counts": table_name(settings, "failed_attempt_line_counts"),
    }
    for table, schema in (
        (tables["runs"], RUN_SCHEMA),
        (tables["telemetry"], TELEMETRY_SCHEMA),
        (tables["line_counts"], LINE_COUNT_SCHEMA),
        (tables["failed_telemetry"], TELEMETRY_SCHEMA),
        (tables["failed_line_counts"], LINE_COUNT_SCHEMA),
    ):
        spark_session.createDataFrame([], schema).write.format("delta").mode("ignore").saveAsTable(table)
    return tables


def load_run(tables: dict[str, str], run_id: str) -> dict[str, Any] | None:
    rows = (
        spark_session.table(tables["runs"])
        .where(F.col("run_id") == run_id)
        .limit(2)
        .collect()
    )
    if len(rows) > 1:
        raise RuntimeError(f"Run ledger contains duplicate rows for RUN_ID={run_id!r}")
    return rows[0].asDict(recursive=True) if rows else None


def assert_same_execution(existing: dict[str, Any], input_sha256: str, config_sha256: str) -> None:
    if existing["input_sha256"] != input_sha256 or existing["config_sha256"] != config_sha256:
        raise RuntimeError(
            "RUN_ID already belongs to a different input or configuration; "
            "use a new stable run ID for changed work"
        )


def merge_run_row(tables: dict[str, str], row: dict[str, Any], *, preserve_success: bool) -> None:
    source = spark_session.createDataFrame([row], RUN_SCHEMA)
    merge = DeltaTable.forName(spark_session, tables["runs"]).alias("t").merge(
        source.alias("s"),
        "t.run_id = s.run_id",
    )
    same_execution = (
        "t.input_sha256 = s.input_sha256 AND "
        "t.config_sha256 = s.config_sha256"
    )
    if preserve_success:
        merge = merge.whenMatchedUpdateAll(
            condition=f"{same_execution} AND t.status <> 'SUCCEEDED'"
        )
    else:
        merge = merge.whenMatchedUpdateAll(condition=same_execution)
    merge.whenNotMatchedInsertAll().execute()


def replace_scope_rows(
    table: str,
    frame: DataFrame,
    key_columns: tuple[str, ...],
    scope_column: str,
    scope_value: str,
) -> None:
    condition = " AND ".join(f"t.`{column}` = s.`{column}`" for column in key_columns)
    (
        DeltaTable.forName(spark_session, table)
        .alias("t")
        .merge(frame.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete(
            condition=F.col(f"t.{scope_column}") == F.lit(scope_value)
        )
        .execute()
    )

## 4. Stage, execute, and commit

This is the activity cell. It uses a fresh SDK configuration for every attempt. If inference or either authoritative merge fails, the partial initialized SDK result is written to failed-attempt tables and the original activity still fails.

In [ ]:
@dataclass(frozen=True)
class StagedVideo:
    path: Path
    temporary_directory: Path | None


def stage_video(video_uri: str, attempt_id: str) -> StagedVideo:
    local_path = Path(video_uri)
    if local_path.is_file():
        return StagedVideo(local_path, None)

    parsed = urlparse(video_uri)
    if parsed.scheme not in {"abfs", "abfss"}:
        raise FileNotFoundError(
            f"Video is not a readable local/Lakehouse path and is not an abfs URI: {video_uri}"
        )

    temporary_directory = Path("/tmp") / f"people-counter-{attempt_id}"
    temporary_directory.mkdir(mode=0o700, parents=False, exist_ok=False)
    suffix = Path(parsed.path).suffix or ".video"
    staged_path = temporary_directory / f"input{suffix}"
    notebookutils = importlib.import_module("notebookutils")
    try:
        copied = notebookutils.fs.cp(video_uri, staged_path.as_uri())
        if copied is False or not staged_path.is_file():
            raise RuntimeError(f"Fabric could not stage {video_uri}")
    except Exception:
        staged_path.unlink(missing_ok=True)
        temporary_directory.rmdir()
        raise
    return StagedVideo(staged_path, temporary_directory)


def cleanup_staged_video(staged: StagedVideo | None) -> None:
    if staged is None or staged.temporary_directory is None:
        return
    staged.path.unlink(missing_ok=True)
    staged.temporary_directory.rmdir()


def build_config(settings: Settings, video: Path):
    last_reported = {"frames": -1}

    def report_progress(current) -> None:
        step = max(1, current.total_sampled_frames // 20)
        if (
            current.processed_frames == 0
            or current.processed_frames == current.total_sampled_frames
            or current.processed_frames - last_reported["frames"] >= step
        ):
            print(
                f"run={settings.run_id} processed="
                f"{current.processed_frames}/{current.total_sampled_frames}"
            )
            last_reported["frames"] = current.processed_frames

    common = {
        "video": video,
        "device_variant": settings.device_variant,
        "device": settings.device,
        "batch_size": settings.batch_size,
        "sample_fps": settings.sample_fps,
        "detection_threshold": settings.detection_threshold,
        "use_fp16": settings.use_fp16,
        "line": settings.line,
        "progress_callback": report_progress,
    }
    if settings.pipeline == "rtdetr-osnet":
        return RTDetrOsnetConfig(**common, detector_model=settings.detector_model)
    return RFDetrBotsortConfig(
        **common,
        camera_motion_compensation=settings.camera_motion_compensation,
    )


def provenance(
    settings: Settings,
    attempt_id: str,
    source_video: str,
    input_sha256: str,
    config_sha256: str,
    recorded_at: datetime,
) -> dict[str, Any]:
    return {
        "run_id": settings.run_id,
        "attempt_id": attempt_id,
        "source_uri": settings.video_uri,
        "source_video": source_video,
        "input_sha256": input_sha256,
        "config_sha256": config_sha256,
        "recorded_at": recorded_at,
    }


def result_frames(
    settings: Settings,
    attempt_id: str,
    source_video: str,
    input_sha256: str,
    config_sha256: str,
    result,
) -> tuple[DataFrame, DataFrame]:
    common = provenance(
        settings,
        attempt_id,
        source_video,
        input_sha256,
        config_sha256,
        utc_now(),
    )
    telemetry = [{**common, **record} for record in telemetry_records(result)]
    line_counts = [{**common, **record} for record in line_count_records(result)]
    return (
        spark_session.createDataFrame(telemetry, TELEMETRY_SCHEMA),
        spark_session.createDataFrame(line_counts, LINE_COUNT_SCHEMA),
    )


def run_row(
    settings: Settings,
    attempt_id: str,
    source_video: str,
    input_sha256: str,
    config_json: str,
    config_sha256: str,
    status: str,
    started_at: datetime,
    result=None,
    error: Exception | None = None,
) -> dict[str, Any]:
    completed = utc_now() if status in {"SUCCEEDED", "FAILED"} else None
    return {
        "run_id": settings.run_id,
        "attempt_id": attempt_id,
        "source_uri": settings.video_uri,
        "source_video": source_video,
        "input_sha256": input_sha256,
        "config_json": config_json,
        "config_sha256": config_sha256,
        "status": status,
        "started_at": started_at,
        "completed_at": completed,
        "distinct_people": len(result.telemetry) if result is not None else None,
        "line_in_count": result.line_in_count if result is not None else None,
        "line_out_count": result.line_out_count if result is not None else None,
        "processed_frames": result.processed_frames if result is not None else None,
        "ended_early": result.ended_early if result is not None else None,
        "error_type": type(error).__name__ if error is not None else None,
        "error_message": str(error)[:4000] if error is not None else None,
    }


settings = load_settings()
tables = ensure_tables(settings)
attempt_id = uuid.uuid4().hex
started_at = utc_now()
staged = None
config = None
ledger_claimed = False
activity_summary = None

try:
    staged = stage_video(settings.video_uri, attempt_id)
    source_video = Path(urlparse(settings.video_uri).path).name or staged.path.name
    input_sha256 = sha256_file(staged.path)
    config_json, config_sha256 = sha256_json(settings.canonical_config())

    existing = load_run(tables, settings.run_id)
    if existing is not None:
        assert_same_execution(existing, input_sha256, config_sha256)

    if existing is not None and existing["status"] == "SUCCEEDED":
        activity_summary = existing
        print(f"RUN_ID={settings.run_id!r} already succeeded; inference skipped")
    else:
        merge_run_row(
            tables,
            run_row(
                settings,
                attempt_id,
                source_video,
                input_sha256,
                config_json,
                config_sha256,
                "RUNNING",
                started_at,
            ),
            preserve_success=True,
        )
        claimed = load_run(tables, settings.run_id)
        if claimed is None:
            raise RuntimeError("Run ledger claim was not persisted")
        assert_same_execution(claimed, input_sha256, config_sha256)

        if claimed["status"] == "SUCCEEDED":
            activity_summary = claimed
            print(f"A concurrent attempt completed RUN_ID={settings.run_id!r}; inference skipped")
        else:
            ledger_claimed = True
            config = build_config(settings, staged.path)
            result = run(config)
            telemetry_frame, line_count_frame = result_frames(
                settings,
                attempt_id,
                source_video,
                input_sha256,
                config_sha256,
                result,
            )
            replace_scope_rows(
                tables["telemetry"],
                telemetry_frame,
                ("run_id", "person_id"),
                "run_id",
                settings.run_id,
            )
            replace_scope_rows(
                tables["line_counts"],
                line_count_frame,
                ("run_id", "frame"),
                "run_id",
                settings.run_id,
            )
            success_row = run_row(
                settings,
                attempt_id,
                source_video,
                input_sha256,
                config_json,
                config_sha256,
                "SUCCEEDED",
                started_at,
                result=result,
            )
            merge_run_row(tables, success_row, preserve_success=False)
            activity_summary = success_row
except Exception as activity_error:
    try:
        if config is not None and config.result.initialized:
            failed_telemetry, failed_line_counts = result_frames(
                settings,
                attempt_id,
                source_video,
                input_sha256,
                config_sha256,
                config.result,
            )
            replace_scope_rows(
                tables["failed_telemetry"],
                failed_telemetry,
                ("attempt_id", "person_id"),
                "attempt_id",
                attempt_id,
            )
            replace_scope_rows(
                tables["failed_line_counts"],
                failed_line_counts,
                ("attempt_id", "frame"),
                "attempt_id",
                attempt_id,
            )
        if ledger_claimed:
            merge_run_row(
                tables,
                run_row(
                    settings,
                    attempt_id,
                    source_video,
                    input_sha256,
                    config_json,
                    config_sha256,
                    "FAILED",
                    started_at,
                    result=config.result if config is not None and config.result.initialized else None,
                    error=activity_error,
                ),
                preserve_success=True,
            )
    except Exception as recording_error:
        raise RuntimeError(
            "The activity failed and its diagnostic persistence also failed"
        ) from recording_error
    raise
finally:
    cleanup_staged_video(staged)

display(activity_summary)

## 5. Verify the committed run

These reads are scoped to `RUN_ID`; they do not collect the full output tables. A successful run has one `SUCCEEDED` ledger row, and rerunning the notebook with the same input and configuration returns that row without loading models.

In [ ]:
verified_run = (
    spark_session.table(tables["runs"])
    .where(F.col("run_id") == settings.run_id)
    .select(
        "run_id",
        "attempt_id",
        "status",
        "source_video",
        "distinct_people",
        "line_in_count",
        "line_out_count",
        "processed_frames",
        "ended_early",
        "completed_at",
    )
)
verified_run.show(truncate=False)

if verified_run.where(F.col("status") == "SUCCEEDED").count() != 1:
    raise RuntimeError("Expected exactly one SUCCEEDED ledger row")

print(
    "Authoritative rows:",
    {
        "telemetry": spark_session.table(tables["telemetry"]).where(F.col("run_id") == settings.run_id).count(),
        "line_counts": spark_session.table(tables["line_counts"]).where(F.col("run_id") == settings.run_id).count(),
    },
)